# Pre-encoded BEHAVIOR Dataset Tests

Validates that `quastAI/behavior2025-vjepa2-vitg-130h-demo-embeddings` (MDS shards) was built
correctly from the base dataset.
**What these tests cannot guarantee:**
- Visual quality of decoded video frames (pixel correctness is not checked)
- Whether V-JEPA2 produced semantically meaningful embeddings
- Correctness for episodes not sampled in the spot-checks (Tests 4 & 5)

In [ ]:
# pip install huggingface_hub mosaicml-streaming pandas numpy pyarrow opencv-python

import json
import os
from collections import defaultdict
from math import ceil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from streaming import StreamingDataset

# ── Config — must match configs/behavior-vitg16-256px-16f.yaml ───────────────
HF_REPO_ID    = "quastAI/behavior2025-vjepa2-vitg-130h-demo-embeddings"
HF_SUBFOLDER  = "shards_256px_vit_16_g"
BASE_DATASET  = Path("/Users/julianquast/Documents/Documents/LOTO-H-JEPA/dataset_pipeline/base_dataset")
MANIFEST_PATH = BASE_DATASET / "manifest.json"

TARGET_FPS          = 5
FPC                 = 8
TEMPORAL_PATCH_SIZE = 2          # tubelet size; must match model config
FRAMES_PER_CLIP     = FPC                          # = 8 sampled frames per window
STATE_DIM           = 133        # curated proprio vector (see BehaviorVideoDataset.PROPRIO_SLICES)
ACTION_DIM          = 23
NATIVE_FPS          = 30
FSTP_NOM            = ceil(NATIVE_FPS / TARGET_FPS)  # = 6 native frames per sampled step
# Each MDS row is one sampled frame; actions cover fstp native frames from that frame onward.
ACTIONS_PER_ROW     = FSTP_NOM * ACTION_DIM  # = 138
CAMERA_VIEW         = "multi"    # "head" or "multi" — must match config

views = ["head"] if CAMERA_VIEW == "head" else ["head", "left_wrist", "right_wrist"]

# Mirrors BehaviorVideoDataset.PROPRIO_SLICES — used in state alignment test.
PROPRIO_SLICES = [
    np.s_[6:28],    # joint_qpos controllable
    np.s_[34:56],   # joint_qpos_sin controllable
    np.s_[62:84],   # joint_qpos_cos controllable
    np.s_[84:112],  # joint_qvel (all)
    np.s_[152:158], # robot_lin_vel + robot_ang_vel
    np.s_[186:197], # eef_left_pos + quat + gripper
    np.s_[225:244], # eef_right_pos + quat + gripper + trunk
    np.s_[253:256], # base_qvel
]

def extract_proprio(full_states: np.ndarray) -> np.ndarray:
    """Extract 133-dim curated proprio from a (..., 256) state array."""
    return np.concatenate([full_states[..., s] for s in PROPRIO_SLICES], axis=-1)

N_SPOT_CHECK = 5   # episodes cross-checked against raw parquet

print(f"FSTP_NOMINAL={FSTP_NOM}  FRAMES_PER_CLIP={FRAMES_PER_CLIP}  TEMPORAL_PATCH_SIZE={TEMPORAL_PATCH_SIZE}")
print(f"STATE_DIM={STATE_DIM}  ACTIONS_PER_ROW={ACTIONS_PER_ROW}")
print(f"CAMERA_VIEW={CAMERA_VIEW}  views={views}")

---
## Test 1 — Download & Schema Validation

**Guarantees:** The HF repo can be downloaded; `index.json` is valid; every shard contains all
expected MDS columns; the dataset is non-empty; first-row array shapes match pipeline config.

**Does NOT guarantee:** Numerical correctness of any values.

In [ ]:
import os
import random

os.environ["HF_TOKEN"] = ""
# hf_transfer uses a Rust-based downloader; significantly faster on Colab.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.system("pip install -q hf_transfer")

print("Downloading encoded shards from Hugging Face …")
local_repo = Path(
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        allow_patterns=[f"{HF_SUBFOLDER}/**"],
        max_workers=16,
    )
)
encoded_dir = local_repo / HF_SUBFOLDER
print(f"Local path: {encoded_dir}")
assert encoded_dir.exists(), f"Expected directory not found: {encoded_dir}"

index_path = encoded_dir / "index.json"
assert index_path.exists(), "index.json missing — dataset may be incomplete"
with open(index_path) as f:
    index = json.load(f)

shards = index.get("shards", [])
assert len(shards) > 0, "index.json contains no shards"
total_rows = sum(s["samples"] for s in shards)
print(f"Shards: {len(shards)},  total rows (sampled frames): {total_rows:,}")
assert total_rows > 0, "Dataset is empty"

TOKEN_COLS = {f"tokens_{v}" for v in views}
EXPECTED_COLUMNS = TOKEN_COLS | {"actions", "states", "cam_rel_poses", "frame_index",
                                  "episode_idx", "sample_idx", "step_pos", "episode_len"}
for shard_info in shards:
    cols = set(shard_info.get("column_names", []))
    missing = EXPECTED_COLUMNS - cols
    assert not missing, f"Shard {shard_info['raw_data']['basename']} missing columns: {missing}"

print("Schema OK.  Streaming all rows …")
dataset = StreamingDataset(local=str(encoded_dir), shuffle=False)

FINITENESS_SAMPLE = 500
finiteness_check_indices = set(random.sample(range(len(dataset)), min(FINITENESS_SAMPLE, len(dataset))))

TOKEN_SHAPE = None
shape_failures, nan_rows = [], []
rows_meta = []

for i in range(len(dataset)):
    row = dataset[i]
    tok = row[f"tokens_{views[0]}"]

    assert isinstance(tok, np.ndarray) and tok.ndim == 2, \
        f"row {i}: unexpected token type/shape for tokens_{views[0]}"

    if TOKEN_SHAPE is None:
        TOKEN_SHAPE = tok.shape
        for v in views[1:]:
            assert row[f"tokens_{v}"].shape == TOKEN_SHAPE, \
                f"row {i}: tokens_{v} shape {row[f'tokens_{v}'].shape} != {TOKEN_SHAPE}"
        assert isinstance(row["actions"], np.ndarray) and row["actions"].ndim == 1, \
            f"row {i}: actions not a 1-D array"
        assert row["actions"].shape[0] == ACTIONS_PER_ROW, \
            f"actions: expected {ACTIONS_PER_ROW} (fstp={FSTP_NOM}×{ACTION_DIM}), " \
            f"got {row['actions'].shape[0]}"
        assert isinstance(row["states"], np.ndarray) and row["states"].ndim == 1
        assert row["states"].shape[0] == STATE_DIM, \
            f"states: expected {STATE_DIM}, got {row['states'].shape[0]}"
        assert isinstance(row["cam_rel_poses"], np.ndarray)
        assert row["cam_rel_poses"].shape[0] == 21, \
            f"cam_rel_poses: expected 21, got {row['cam_rel_poses'].shape[0]}"
    else:
        # Check shape for all views on every subsequent row
        for v in views:
            t = row[f"tokens_{v}"]
            if t.shape != TOKEN_SHAPE:
                shape_failures.append(f"row {i} tokens_{v}: expected {TOKEN_SHAPE}, got {t.shape}")

    if i in finiteness_check_indices:
        for v in views:
            if not np.all(np.isfinite(row[f"tokens_{v}"])):
                nan_rows.append((i, v))

    rows_meta.append({
        "episode_idx": row["episode_idx"],
        "sample_idx":  row["sample_idx"],
        "step_pos":    row["step_pos"],
        "frame_index": row["frame_index"],
        "episode_len": row["episode_len"],
        "actions":     row["actions"],
        "states":      row["states"],
    })

print(f"Streamed {len(rows_meta):,} rows.")
print(f"Token shape per sampled frame: {TOKEN_SHAPE}  "
      f"(tokens_per_frame={TOKEN_SHAPE[0]}, hidden_dim={TOKEN_SHAPE[1]})")
print("PASS Test 1")

---
## Test 2 — Episode Completeness

**Guarantees:** For every episode: `frame_pos` is exactly `0 … episode_len-1` with no gaps and
no duplicates, and `episode_len` is consistent across all rows of that episode.  Rules out lost
or duplicated frames during the episode-accumulation / flush phase in `BehaviorEpisodePreencoder`.

**Does NOT guarantee:** That `episode_len` equals the expected number of sampled frames from the
source video (see Test 3 for that).

In [ ]:
episodes: dict = defaultdict(list)
for row in rows_meta:
    episodes[row["episode_idx"]].append(row)

print(f"Unique episodes in MDS: {len(episodes)}")

failures = []
for ep_id, ep_rows in episodes.items():
    stored_len = ep_rows[0]["episode_len"]
    if not all(r["episode_len"] == stored_len for r in ep_rows):
        failures.append(f"ep {ep_id}: inconsistent episode_len values")
        continue
    if len(ep_rows) != stored_len:
        failures.append(f"ep {ep_id}: stored episode_len={stored_len} but found {len(ep_rows)} rows")
        continue
    positions = sorted(r["step_pos"] for r in ep_rows)
    if positions != list(range(stored_len)):
        failures.append(f"ep {ep_id}: step_pos is not 0..{stored_len-1}")

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} episode completeness failures")

print(f"PASS Test 2 — all {len(episodes)} episodes have contiguous step_pos with correct episode_len.")


---
## Test 3 — Frame-Index Sampling Pattern

**Guarantees:** For every episode, stored `frame_index` values are strictly monotonically increasing
and the step between consecutive frames equals `ceil(video_fps / TARGET_FPS)`.  Confirms the video
was subsampled at exactly `TARGET_FPS=5`, not densely or randomly.

**Does NOT guarantee:** That the decoded pixel content of those frames is correct.

In [ ]:
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

_VIEW_IDX = {"head": 0, "left_wrist": 1, "right_wrist": 2}

def video_path(sample_idx, view="head"):
    ep = manifest["episodes"][sample_idx]
    vfiles = ep.get("video_files") or [ep["video_file"]]
    idx = _VIEW_IDX.get(view, 0)
    name = Path(vfiles[min(idx, len(vfiles) - 1)]).name
    return BASE_DATASET / ep["task_name"] / "video" / view / name

def parquet_path(sample_idx):
    ep = manifest["episodes"][sample_idx]
    name = os.path.splitext(os.path.basename(ep["episode_file"]))[0]
    return BASE_DATASET / ep["task_name"] / "data" / f"{name}.parquet"

failures = []
for ep_id, ep_rows in episodes.items():
    sorted_rows = sorted(ep_rows, key=lambda r: r["step_pos"])
    fis = [r["frame_index"] for r in sorted_rows]

    if any(fis[i] >= fis[i+1] for i in range(len(fis)-1)):
        failures.append(f"ep {ep_id}: frame_index not strictly increasing")
        continue

    vp = video_path(sorted_rows[0]["sample_idx"])
    if not vp.exists():
        print(f"  SKIP ep {ep_id}: video not found locally")
        continue
    cap = cv2.VideoCapture(str(vp))
    fstp = max(1, ceil(cap.get(cv2.CAP_PROP_FPS) / TARGET_FPS))
    cap.release()

    expected_step = fstp
    steps = [fis[i+1] - fis[i] for i in range(len(fis)-1)]
    wrong = [s for s in steps if s != expected_step]
    if wrong:
        failures.append(
            f"ep {ep_id} (fstp={fstp}, expected_step={expected_step}): "
            f"{len(wrong)}/{len(steps)} steps != {expected_step}: e.g. {wrong[:5]}"
        )

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} sampling-pattern failures")

print(f"PASS Test 3 — all episodes have monotonically increasing frame_index "
      f"with step == fstp = {FSTP_NOM}.")

---
## Test 4 — State Alignment (cross-check vs. base parquet)

**Guarantees:** For `N_SPOT_CHECK` randomly chosen episodes, the `states` vector at every sampled
frame exactly matches `extract_proprio(parquet['observation.state'][frame_index])` — the
proprioceptive state AT the sampled frame.  Rules out off-by-one index errors, wrong proprio
slice, and wrong episode→parquet mapping.

**Does NOT guarantee:** Correctness for episodes not in the spot-check sample.

In [ ]:
rng = np.random.default_rng(42)
check_ids = rng.choice(list(episodes.keys()), size=min(N_SPOT_CHECK, len(episodes)), replace=False)

state_failures = []
for ep_id in check_ids:
    sorted_rows = sorted(episodes[ep_id], key=lambda r: r["step_pos"])
    pp = parquet_path(sorted_rows[0]["sample_idx"])
    if not pp.exists():
        print(f"  SKIP ep {ep_id}: parquet not found")
        continue

    df = pd.read_parquet(pp, columns=["observation.state"])
    raw = np.asarray(df["observation.state"].to_list(), dtype=np.float32)  # (T, 256)

    # Determine fstp for this episode from the video.
    vp = video_path(sorted_rows[0]["sample_idx"])
    if vp.exists():
        cap = cv2.VideoCapture(str(vp))
        vfps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        fstp = max(1, ceil(vfps / TARGET_FPS))
    else:
        fstp = FSTP_NOM  # fall back to nominal

    for row in sorted_rows:
        fi = row["frame_index"]  # native frame index of the sampled frame
        # States are stored AT the sampled frame.
        if fi >= len(raw):
            # Padded window at episode end — state was zeroed out in loadvideo_decord.
            expected = np.zeros(STATE_DIM, dtype=np.float32)
        else:
            expected = extract_proprio(raw[fi : fi + 1]).flatten()

        if not np.allclose(expected, row["states"], rtol=1e-4, atol=1e-5):
            state_failures.append(
                f"ep {ep_id} step_pos={row['step_pos']} fi={fi}: "
                f"max_diff={np.max(np.abs(expected - row['states'])):.2e}"
            )
    print(f"  ep {ep_id}: {len(sorted_rows)} steps checked — OK")

if state_failures:
    for f in state_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(state_failures)} state alignment failures")

print("PASS Test 4 — states == extract_proprio(parquet[frame_index]) for all spot-checked episodes.")

---
## Test 5 — Action Aggregation (cross-check vs. base parquet)

**Guarantees:** For `N_SPOT_CHECK` episodes, `actions` at each sampled frame equals
`flatten(raw_actions[frame_index : frame_index + fstp, :ACTION_DIM])` — the `fstp` native
actions executed from that sampled frame onward (causally aligned with the stored `states`).
Verifies correct frame indexing and end-of-episode padding.

**Does NOT guarantee:** Correctness for episodes outside the spot-check.

In [ ]:
def reconstruct_action_chunk(raw_act, start, fstp, max_len):
    """Reconstruct one sampled-frame action chunk matching loadvideo_decord's logic."""
    if start >= max_len:
        return np.zeros((fstp, ACTION_DIM), dtype=np.float32)
    end = min(start + fstp, max_len)
    chunk = raw_act[start:end]
    if len(chunk) == 0:
        return np.zeros((fstp, ACTION_DIM), dtype=np.float32)
    if len(chunk) < fstp:
        chunk = np.concatenate([chunk, np.repeat(chunk[-1:], fstp - len(chunk), axis=0)])
    return chunk[:fstp]


action_failures = []
for ep_id in check_ids:
    sorted_rows = sorted(episodes[ep_id], key=lambda r: r["step_pos"])
    sid = sorted_rows[0]["sample_idx"]
    vp, pp = video_path(sid), parquet_path(sid)

    if not vp.exists() or not pp.exists():
        print(f"  SKIP ep {ep_id}: video or parquet not found")
        continue

    cap = cv2.VideoCapture(str(vp))
    vfps = cap.get(cv2.CAP_PROP_FPS)
    n_vid = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    fstp = max(1, ceil(vfps / TARGET_FPS))

    df = pd.read_parquet(pp, columns=["action"])
    raw_act = np.asarray(df["action"].to_list(), dtype=np.float32)[:, :ACTION_DIM]
    max_len = min(n_vid, len(raw_act))

    for row in sorted_rows:
        fi = row["frame_index"]  # native frame index of the sampled frame

        # Actions cover fstp native frames from fi onward.
        expected = reconstruct_action_chunk(raw_act, fi, fstp, max_len).reshape(ACTIONS_PER_ROW)

        if not np.allclose(expected, row["actions"], rtol=1e-3, atol=1e-4):
            action_failures.append(
                f"ep {ep_id} step_pos={row['step_pos']} fi={fi}: "
                f"max_diff={np.max(np.abs(expected - row['actions'])):.2e}"
            )
    print(f"  ep {ep_id}: {len(sorted_rows)} action chunks checked — OK")

if action_failures:
    for f in action_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(action_failures)} action aggregation failures")

print(f"PASS Test 5 — actions == flatten(raw_actions[frame_index : frame_index+fstp]) for all spot-checked episodes.")

---
## Test 6 — Token Shape Consistency & Finiteness

**Guarantees:** Every row has the *same* `(tokens_per_frame, hidden_dim)` shape (no mixed-shard
accidents) and a random sample of ~500 rows (~0.5%) are checked for NaN/Inf from fp16 overflow
or encoder crash. Shape is checked for all rows; finiteness is sampled to keep runtime tractable.

**Does NOT guarantee:** Semantic quality of embeddings, or that bfloat16→float16 conversion did
not cause silent precision loss.

In [ ]:
# shape_failures and nan_rows were collected during the streaming pass in Test 1
if shape_failures:
    for f in shape_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(shape_failures)} token shape mismatches")

assert TOKEN_SHAPE[0] > 0 and TOKEN_SHAPE[1] > 0, "Degenerate token shape"

print(f"Token shape: {TOKEN_SHAPE}")
if nan_rows:
    print(f"WARNING — {len(nan_rows)} / {len(rows_meta) * len(views)} view-rows contain NaN or Inf in tokens")
    for row_i, view in nan_rows[:5]: print(f"  row {row_i} tokens_{view}")
else:
    print(f"PASS Test 6 — all {len(rows_meta):,} token arrays have shape {TOKEN_SHAPE} "
          f"and are finite across all {len(views)} view(s).")

---
## Test 7 — Episode Coverage vs. Manifest

**Guarantees:** Every episode in the manifest appears in the MDS (no silent skip), and
no extra `sample_idx` values exist that have no corresponding manifest entry.

**Does NOT guarantee:** That the *frames* within each episode are correct (Tests 2–5 cover that).

In [ ]:
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
n_manifest = len(manifest["episodes"])
manifest_indices = set(range(n_manifest))

mds_sample_indices = {row["sample_idx"] for row in rows_meta}

missing = manifest_indices - mds_sample_indices
extra   = mds_sample_indices - manifest_indices

if missing:
    for idx in sorted(missing):
        ep = manifest["episodes"][idx]
        print(f"FAIL: sample_idx={idx} ({ep['task_name']}) missing from MDS")
    raise AssertionError(f"{len(missing)} manifest episodes missing from MDS")

if extra:
    raise AssertionError(f"MDS contains sample_idx values not in manifest: {sorted(extra)}")

print(f"PASS Test 7 — all {n_manifest} manifest episodes present in MDS, no extras.")


---
## Test 8 — Episode Frame Count vs. Expected Sampled Count

Each MDS row is one sampled frame, so `episode_len` must equal exactly the number of sampled
frames `BehaviorVideoDataset._episode_sampled_indices` would produce for that episode.

**Guarantees:** For every episode, `episode_len` in the MDS equals
`len(np.arange(0, min(n_video_frames, parquet_len), fstp))`.  Rules out an entire window
being silently dropped or duplicated during the accumulate/flush phase.

**Does NOT guarantee:** That the pixel content of each frame is correct.

In [ ]:
from math import ceil

failures = []
for ep_id, ep_rows in episodes.items():
    stored_len = ep_rows[0]["episode_len"]
    sid = ep_rows[0]["sample_idx"]

    vp = video_path(sid)
    pp = parquet_path(sid)
    if not vp.exists() or not pp.exists():
        print(f"  SKIP ep {ep_id}: files not found locally")
        continue

    cap = cv2.VideoCapture(str(vp))
    vfps = cap.get(cv2.CAP_PROP_FPS)
    n_vid = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    fstp = max(1, ceil(vfps / TARGET_FPS))

    parquet_len = len(pd.read_parquet(pp, columns=["action"]))
    max_len = min(n_vid, parquet_len)

    # Sampled frame indices — same as _episode_sampled_indices.
    sampled = np.arange(0, max_len, fstp, dtype=np.int64)
    # Each sampled frame becomes one MDS row.
    expected_steps = len(sampled)

    if stored_len != expected_steps:
        failures.append(
            f"ep {ep_id} (sample_idx={sid}): "
            f"stored episode_len={stored_len} frames, expected={expected_steps} "
            f"(n_vid={n_vid}, parquet_len={parquet_len}, fstp={fstp})"
        )
    else:
        print(f"  ep {ep_id}: episode_len={stored_len} steps == expected {expected_steps}  OK")

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} episode step-count mismatches")

print("PASS Test 8 — all episode frame counts match len(np.arange(0, max_len, fstp)).")


---
## Final Summary

Each MDS row is one **sampled frame** (encoded independently as a fake `TEMPORAL_PATCH_SIZE`-frame
tubelet to satisfy V-JEPA2's input contract).
- `tokens_<view>` → `(tokens_per_frame, embed_dim)` — V-JEPA2 tokens for one sampled frame per camera view
- `actions`       → `(fstp × action_dim,)` = `(138,)` — native-fps actions from `frame_index` onward
- `states`        → `(133,)` — curated proprio AT `frame_index` (the observed robot state)
- `cam_rel_poses` → `(21,)` — 3 cameras × (pos[3]+quat[4]) AT `frame_index`
- `frame_index`   → native video frame index of the sampled frame
- `step_pos`      → position within episode (0-indexed)
- `episode_len`   → total sampled frames in episode = `len(np.arange(0, max_len, fstp))`

| Test | What it guarantees | Remaining gap |
|------|--------------------|---------------|
| 1 Schema | All MDS columns present; correct per-frame shapes for tokens/actions/states/cam_rel_poses | Not numerical values |
| 2 Completeness | `step_pos` is `0…episode_len-1` with no gaps or duplicates | Not whether episode count matches manifest |
| 3 Sampling | `frame_index` increases by `fstp` between consecutive rows | Not pixel content of decoded frames |
| 4 States | `states == extract_proprio(parquet[frame_index])` | Only N_SPOT_CHECK episodes |
| 5 Actions | `actions == raw_actions[frame_index : frame_index+fstp, :ACTION_DIM].flatten()` | Only N_SPOT_CHECK episodes |
| 6 Tokens | Consistent shape, all finite (incl. bf16→fp16 overflow) | Semantic quality of embeddings |
| 7 Coverage | Every manifest episode present in MDS, no extras | Frame-level correctness within episode |
| 8 Frame count | `episode_len == len(np.arange(0, max_len, fstp))` | Pixel content; window ordering within episode |

**Fundamentally untestable here:** pixel correctness and encoder output quality — both
require re-running the full GPU inference pipeline.